# Laboratoire 3 : Encodage de données et feature maps

**Quantum Machine Learning — Master/PhD**

Semaine 4 — Encodage de données classiques en états quantiques

## Objectifs

- Implémenter et comparer les stratégies d'encodage quantique
- Comprendre AngleEmbedding, AmplitudeEmbedding, IQPEmbedding (PennyLane)
- Implémenter ZZFeatureMap et PauliFeatureMap (Qiskit)
- Analyser le coût en ressources et l'expressivité de chaque méthode

## Bibliothèques requises

- `pennylane` (≥ 0.45)
- `qiskit` (≥ 2.0)
- `qiskit-machine-learning`
- `matplotlib`
- `numpy`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import pennylane as qml
from pennylane import numpy as pnp

from qiskit import QuantumCircuit
from qiskit.circuit.library import ZZFeatureMap, PauliFeatureMap, ZFeatureMap
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector

print('Bibliothèques chargées avec succès.')
print(f'PennyLane : {qml.__version__}')

---
## Partie A : AngleEmbedding (PennyLane)

L'AngleEmbedding encode chaque feature comme un angle de rotation :

$$x \in \mathbb{R}^n \mapsto \bigotimes_{i=1}^{n} R(x_i) |0\rangle$$

Chaque feature $x_i$ est encodée via une porte $R_X$, $R_Y$ ou $R_Z$ sur un qubit dédié.

In [ ]:
# AngleEmbedding avec PennyLane

n_qubits = 4
dev = qml.device('default.qubit', wires=n_qubits)

@qml.qnode(dev)
def circuit_angle_embedding(x, rotation='X'):
    """Angle embedding avec rotation choisie."""
    if rotation == 'X':
        qml.AngleEmbedding(x, wires=range(n_qubits), rotation='X')
    elif rotation == 'Y':
        qml.AngleEmbedding(x, wires=range(n_qubits), rotation='Y')
    elif rotation == 'Z':
        qml.AngleEmbedding(x, wires=range(n_qubits), rotation='Z')
    return qml.state()

# Données exemple
x_test = np.array([0.5, 1.2, 0.8, 2.0])

print("Données à encoder :", x_test)
print()

for rot in ['X', 'Y', 'Z']:
    psi = circuit_angle_embedding(x_test, rotation=rot)
    print(f"Rotation {rot} : état = {np.round(psi, 3)}")
    print(f"  Probabilités : {np.round(np.abs(psi)**2, 3)}")
    print()

In [ ]:
# Visualisation : nombre de qubits nécessaire

print("=== Ressources pour AngleEmbedding ===")
print(f"Nombre de qubits requis  = nombre de features = {n_qubits}")
print(f"Nombre de portes         = {n_qubits} rotations")
print(f"Profondeur du circuit    = 1")
print()

# On peut aussi visualiser le circuit
print("Circuit pour AngleEmbedding (rotation=X) :")
print(qml.draw(circuit_angle_embedding)(x_test, rotation='X'))

In [ ]:
# Analyse : espace des features avec AngleEmbedding

@qml.qnode(dev)
def etat_angle(x):
    qml.AngleEmbedding(x, wires=range(n_qubits))
    return qml.state()

# Deux points proches dans l'espace d'origine vs. dans l'espace de Hilbert
x1 = np.array([0.1, 0.2, 0.3, 0.4])
x2 = np.array([0.1, 0.2, 0.3, 0.5])  # seule la dernière feature diffère

psi1 = etat_angle(x1)
psi2 = etat_angle(x2)

fidelite = np.abs(np.dot(np.conj(psi1), psi2))**2
distance_euclid = np.linalg.norm(x1 - x2)
distance_fidelite = np.arccos(np.sqrt(fidelite))

print(f"Distance euclidienne (entrée)    : {distance_euclid:.4f}")
print(f"Fidélité (espace de Hilbert)      : {fidelite:.4f}")
print(f"Distance de Bures (sortie)        : {distance_fidelite:.4f}")

---
## Partie B : AmplitudeEmbedding et IQPEmbedding

### AmplitudeEmbedding

Encode directement les amplitudes dans l'état quantique :

$$x \in \mathbb{R}^N \mapsto |\psi_x\rangle = \frac{1}{\|x\|} \sum_{i=1}^{N} x_i |i\rangle$$

### IQPEmbedding

Instantaneous Quantum Polynomial embedding : circuit avec portes diagonales et intriquantes.

In [ ]:
# AmplitudeEmbedding

n_amp = 3  # 2^n_amp = 8 amplitudes possibles
dev_amp = qml.device('default.qubit', wires=n_amp)

@qml.qnode(dev_amp)
def circuit_amplitude(x):
    qml.AmplitudeEmbedding(x, wires=range(n_amp), normalize=True)
    return qml.state()

# Données : 8 valeurs (car 2^3 = 8)
x_amp = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8])

psi_amp = circuit_amplitude(x_amp)
print("=== AmplitudeEmbedding ===")
print(f"Données (normalisées) : {np.round(x_amp / np.linalg.norm(x_amp), 3)}")
print(f"État quantique        : {np.round(psi_amp, 3)}")
print(f"Norme de l'état       : {np.linalg.norm(psi_amp):.6f}")
print()

# Circuit
print("Circuit AmplitudeEmbedding :")
print(qml.draw(circuit_amplitude)(x_amp))

In [ ]:
# IQPEmbedding

n_iqp = 4
dev_iqp = qml.device('default.qubit', wires=n_iqp)

@qml.qnode(dev_iqp)
def circuit_iqp(x):
    qml.IQPEmbedding(x, wires=range(n_iqp), n_repeats=2)
    return qml.state()

x_iqp = np.array([0.5, 1.2, 0.8, 2.0])

psi_iqp = circuit_iqp(x_iqp)
print("=== IQPEmbedding ===")
print(f"Données : {x_iqp}")
print(f"État    : {np.round(psi_iqp, 3)}")
print()
print("Circuit IQPEmbedding (n_repeats=2) :")
print(qml.draw(circuit_iqp)(x_iqp))

In [ ]:
# Comparaison des ressources

print("=" * 55)
print(f"{'Méthode':<22} {'#Qubits':<10} {'#Portes':<10} {'Profondeur':<10}")
print("=" * 55)

# AngleEmbedding
print(f"{'AngleEmbedding':<22} {4:<10} {4:<10} {1:<10}")

# AmplitudeEmbedding (pour N features)
N_amp = 8
qubits_amp = int(np.ceil(np.log2(N_amp)))
print(f"{'AmplitudeEmbedding':<22} {qubits_amp:<10} {'complexe':<10} {1:<10}")

# IQPEmbedding
print(f"{'IQPEmbedding':<22} {4:<10} {'2*n*(n-1)/2':<10} {2:<10}")

print()
print("Note : AmplitudeEmbedding nécessite 2^n amplitudes, normalisation automatique.")
print("IQPEmbedding utilise des portes diagonales et des interactions ZZ sur n_repeats répétitions.")

---
## Partie C : Feature maps avec Qiskit (ZZFeatureMap, PauliFeatureMap)

Les *feature maps* de Qiskit implémentent des encodages utilisant des interactions entre qubits.

- **ZFeatureMap** : rotation $R_Z$ individuelle
- **ZZFeatureMap** : rotations $R_Z$ + interactions $R_{ZZ}$ par paires
- **PauliFeatureMap** : rotations et interactions selon des opérateurs de Pauli

In [ ]:
# ZFeatureMap (encodage individuel)

n_features = 4
zfm = ZFeatureMap(feature_dimension=n_features, reps=1)

print("=== ZFeatureMap ===")
print(zfm.decompose().draw(fold=-1))
print(f"\nProfondeur : {zfm.decompose().depth()}")

In [ ]:
# ZZFeatureMap

zzfm = ZZFeatureMap(feature_dimension=n_features, reps=2)

print("=== ZZFeatureMap (reps=2) ===")
print(zzfm.decompose().draw(fold=-1))
print(f"\nNombre de qubits : {zzfm.num_qubits}")
print(f"Profondeur        : {zzfm.decompose().depth()}")

In [ ]:
# PauliFeatureMap

pfm = PauliFeatureMap(
    feature_dimension=n_features,
    reps=2,
    paulis=['Z', 'ZZ', 'XX']  # interactions Z, ZZ et XX
)

print("=== PauliFeatureMap (paulis=['Z', 'ZZ', 'XX'], reps=2) ===")
print(pfm.decompose().draw(fold=-1))
print(f"\nNombre de qubits : {pfm.num_qubits}")
print(f"Profondeur        : {pfm.decompose().depth()}")

In [ ]:
# Visualisation de l'espace des features généré par ZZFeatureMap

# Générons des données 2D et traçons l'état encodé via l'angle de Bloch
def etat_zzfeaturemap(x1, x2, reps=1):
    """Retourne le statevector après ZZFeatureMap."""
    qc = QuantumCircuit(2)
    fm = ZZFeatureMap(feature_dimension=2, reps=reps)
    params = {f'x[0]': x1, f'x[1]': x2}
    bound = fm.assign_parameters(params)
    qc.compose(bound, inplace=True)
    state = Statevector(qc)
    return np.abs(state.data)**2  # probabilités

# Grille de points
xx, yy = np.meshgrid(np.linspace(-np.pi, np.pi, 10), np.linspace(-np.pi, np.pi, 10))
prob_00 = np.zeros_like(xx)

for i in range(len(xx)):
    for j in range(len(xx[0])):
        probs = etat_zzfeaturemap(xx[i, j], yy[i, j])
        prob_00[i, j] = probs[0]  # P(|00>)

plt.figure(figsize=(6, 5))
plt.pcolormesh(xx, yy, prob_00, shading='auto', cmap='viridis')
plt.colorbar(label=r'$P(|00\rangle)$')
plt.xlabel('x[0]')
plt.ylabel('x[1]')
plt.title(r'Espace des features — ZZFeatureMap (reps=1)')
plt.tight_layout()
plt.show()

---
## Partie D : Comparaison des espaces de features

Comparons les quatre méthodes d'encodage sur plusieurs critères :

1. **Nombre de qubits requis**
2. **Nombre de portes**
3. **Expressivité** : quelle richesse de représentation ?
4. **Régularité** : la distance dans l'espace d'entrée est-elle préservée ?

In [ ]:
# Comparaison systématique des ressources

print("=" * 65)
print(f"{'Méthode':<22} {'Qubits':<8} {'Portes/feature':<15} {'Normalisation':<15}")
print("=" * 65)

print(f"{'AngleEmbedding':<22} {'n':<8} {'1':<15} {'Aucune':<15}")
print(f"{'AmplitudeEmbedding':<22} {'log2(N)':<8} {'N':<15} {'L2':<15}")
print(f"{'IQPEmbedding':<22} {'n':<8} {'O(n²)':<15} {'Aucune':<15}")
print(f"{'ZZFeatureMap':<22} {'n':<8} {'O(n²)':<15} {'Aucune':<15}")
print(f"{'PauliFeatureMap':<22} {'n':<8} {'O(n²)':<15} {'Aucune':<15}")

print()
print("Note : n = nombre de features, N = nombre d'amplitudes (2^n)")

In [ ]:
# Richesse de l'espace : mesure par la distribution des fidélités

np.random.seed(42)
n_samples = 50

# On définit 4 devices avec des embeddings différents
devices = {
    'AngleEmbedding': qml.device('default.qubit', wires=4),
    'IQPEmbedding': qml.device('default.qubit', wires=4),
}

def make_qnode(embedding_name, dev):
    if embedding_name == 'AngleEmbedding':
        @qml.qnode(dev)
        def circuit(x):
            qml.AngleEmbedding(x, wires=range(4))
            return qml.state()
    elif embedding_name == 'IQPEmbedding':
        @qml.qnode(dev)
        def circuit(x):
            qml.IQPEmbedding(x, wires=range(4), n_repeats=2)
            return qml.state()
    return circuit

for name, dev in devices.items():
    circuit = make_qnode(name, dev)
    fidelites = []
    for _ in range(n_samples):
        x_a = np.random.uniform(-np.pi, np.pi, 4)
        x_b = x_a + np.random.normal(0, 0.5, 4)
        psi_a = circuit(x_a)
        psi_b = circuit(x_b)
        f = np.abs(np.dot(np.conj(psi_a), psi_b))**2
        fidelites.append(f)
    print(f"=== {name} ===")
    print(f"  Fidélité moyenne : {np.mean(fidelites):.4f}")
    print(f"  Fidélité std     : {np.std(fidelites):.4f}")
    print(f"  Min / Max        : {np.min(fidelites):.4f} / {np.max(fidelites):.4f}")
    print()

In [ ]:
# Visualisation comparative des distributions de fidélité

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (name, dev) in zip(axes, devices.items()):
    circuit = make_qnode(name, dev)
    fidelites = []
    for _ in range(200):
        x_a = np.random.uniform(-np.pi, np.pi, 4)
        x_b = x_a + np.random.normal(0, 0.5, 4)
        psi_a = circuit(x_a)
        psi_b = circuit(x_b)
        f = np.abs(np.dot(np.conj(psi_a), psi_b))**2
        fidelites.append(f)
    ax.hist(fidelites, bins=20, alpha=0.7, edgecolor='k')
    ax.set_title(name)
    ax.set_xlabel('Fidélité')
    ax.set_ylabel('Fréquence')
    ax.axvline(np.mean(fidelites), color='r', ls='--', label=f'Moyenne={np.mean(fidelites):.3f}')
    ax.legend()

plt.tight_layout()
plt.show()

---
## Exercices

### Exercice 1 : AngleEmbedding avec différentes rotations

Comparez l'expressivité d'AngleEmbedding avec rotation='X', 'Y', 'Z'. Quelle rotation donne l'espace des features le plus riche ?

In [ ]:
# Exercice 1
# Comparez les 3 rotations de l'AngleEmbedding

dev_ex1 = qml.device('default.qubit', wires=2)

for rot in ['X', 'Y', 'Z']:
    @qml.qnode(dev_ex1)
    def circuit_rot(x):
        qml.AngleEmbedding(x, wires=[0, 1], rotation=rot)
        return qml.state()

    # Calculez la fidélité moyenne pour cette rotation
    fids = []
    for _ in range(100):
        x1 = np.random.uniform(-np.pi, np.pi, 2)
        x2 = x1 + np.random.normal(0, 0.3, 2)
        state1 = circuit_rot(x1)
        state2 = circuit_rot(x2)
        fids.append(np.abs(np.dot(np.conj(state1), state2))**2)
    print(f"Rotation {rot} : fidélité moyenne = {np.mean(fids):.4f} ± {np.std(fids):.4f}")

### Exercice 2 : Impact du nombre de répétitions (reps) dans IQPEmbedding

Faites varier `n_repeats` de 1 à 4 et analysez l'impact sur l'espace des features.

In [ ]:
# Exercice 2 : impact de n_repeats dans IQPEmbedding

dev_iqp_ex = qml.device('default.qubit', wires=3)

for reps in range(1, 5):
    @qml.qnode(dev_iqp_ex)
    def circuit_iqp_var(x):
        qml.IQPEmbedding(x, wires=range(3), n_repeats=reps)
        return qml.state()

    x_test = np.array([0.3, 1.5, 0.7])
    psi = circuit_iqp_var(x_test)
    print(f"n_repeats={reps} : état = {np.round(psi, 3)}")
    print(f"  Norme L2 = {np.linalg.norm(psi):.4f}")
    print(f"  Profondeur du circuit = {len(qml.draw(circuit_iqp_var)(x_test).split(chr(10)))} lignes")

### Exercice 3 : ZZFeatureMap avec différents angles

Tracez $P(|00\rangle)$ pour un ZZFeatureMap 2-qubits avec reps=1,2,3. Que constatez-vous sur la richesse de la représentation ?

In [ ]:
# Exercice 3
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, reps in enumerate([1, 2, 3]):
    xx, yy = np.meshgrid(np.linspace(-np.pi, np.pi, 15), np.linspace(-np.pi, np.pi, 15))
    prob_00 = np.zeros_like(xx)

    for i in range(len(xx)):
        for j in range(len(xx[0])):
            probs = etat_zzfeaturemap(xx[i, j], yy[i, j], reps=reps)
            prob_00[i, j] = probs[0]

    im = axes[idx].pcolormesh(xx, yy, prob_00, shading='auto', cmap='viridis')
    axes[idx].set_title(f'ZZFeatureMap reps={reps}')
    axes[idx].set_xlabel('x[0]')
    axes[idx].set_ylabel('x[1]')
    plt.colorbar(im, ax=axes[idx])

plt.tight_layout()
plt.show()

---
## Pour aller plus loin

- Essayez `qml.BasisEmbedding` pour l'encodage par base binaire
- Implémentez un `HamiltonianEmbedding` personnalisé
- Comparez l'expressivité par *expressibility quantifiers* (Šuková et al., 2022)
- Références : [SP21] Ch. 4 — *Machine Learning with Quantum Computers*, [Hav19] — *Supervised learning with quantum-enhanced feature spaces*